# Notebook 1 — Pré-processamento das Curvas de Descarga

Este notebook reproduz o **primeiro passo** do pipeline descrito em Severson et al. (2019):
transformar medições brutas de descarga em vetores padronizados de **1.000 pontos**,
que podem ser comparados diretamente entre ciclos e células.

**Dados:** usamos dados **sintéticos** que simulam células LFP/grafite reais.
O dataset real está disponível no repositório original do artigo.

---
**Conceito central:** para comparar dois ciclos, ambos precisam estar na mesma "grade"
de tensão — não importa quantos pontos foram medidos em cada um.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import UnivariateSpline

plt.rcParams.update({
    'figure.dpi': 110,
    'figure.figsize': (11, 4),
    'font.size': 11,
    'axes.grid': True,
    'grid.alpha': 0.3,
})
print('Bibliotecas carregadas.')


## 1. O que é uma Curva de Descarga?

Durante a descarga de uma bateria, medimos continuamente a **tensão** e a
**corrente** ao longo do tempo. Integrando a corrente, obtemos a
**capacidade acumulada** Q (em Ah).

O artigo representa cada ciclo como **Q(V)**: capacidade entregue em cada
nível de tensão.

| Convenção | Eixo X | Eixo Y | Problema |
|-----------|--------|--------|---------|
| Clássica V(Q) | Capacidade | Tensão | Vetores de comprimento diferente entre ciclos |
| **Usada no artigo Q(V)** | Tensão | Capacidade | Faixa de tensão igual para todos os ciclos |

> A faixa de tensão (2,0 V a 3,5 V) é **idêntica** para todos os ciclos de todas
> as células — criando automaticamente uma base comum de comparação.


In [ ]:
def simular_curva_bruta(ciclo, taxa_deg=0.001, semente=42):
    """
    Simula uma curva de descarga bruta (V, Q) de uma celula LFP/grafite.
    ciclo     : numero do ciclo (ex: 10, 100)
    taxa_deg  : velocidade de degradacao da celula
    semente   : para reprodutibilidade
    Retorna V (tensoes decrescentes) e Q (capacidades acumuladas).
    """
    rng = np.random.default_rng(semente + ciclo * 7)
    n_pts = 190 + rng.integers(-25, 40)   # ciclos reais tem 150-250 pontos
    deg = taxa_deg * ciclo

    V = np.linspace(3.5, 2.0, n_pts) + rng.normal(0, 0.003, n_pts)
    V = np.clip(V, 2.0, 3.5)
    V = np.sort(V)[::-1]   # mantem ordem decrescente

    x = (3.5 - V) / 1.5   # progresso normalizado [0, 1]

    # Dois plateaus do LFP/grafite; degradacao LAMdeNE desloca os plateaus
    p1 = 1 / (1 + np.exp(-28 * (x - 0.12 - deg * 0.25)))
    p2 = 1 / (1 + np.exp(-18 * (x - 0.62 - deg * 0.40)))
    p3 = 1 / (1 + np.exp(-12 * (x - 0.92)))

    cap_max = 1.1 * (1 - 0.08 * deg)   # LLI reduz capacidade maxima
    Q = cap_max * (0.42 * p1 + 0.46 * p2 + 0.12 * p3)
    Q += rng.normal(0, 0.0015, n_pts)

    return V, np.clip(Q, 0, None)

V10,  Q10  = simular_curva_bruta(ciclo=10,  taxa_deg=0.002)
V100, Q100 = simular_curva_bruta(ciclo=100, taxa_deg=0.002)

print(f'Ciclo  10: {len(V10):>3} pontos  |  Q_max = {Q10.max():.4f} Ah')
print(f'Ciclo 100: {len(V100):>3} pontos  |  Q_max = {Q100.max():.4f} Ah')
print('Numero de pontos DIFERENTE entre ciclos!')


## 2. O Problema: Ciclos têm Números Diferentes de Pontos

Cada ciclo produz um vetor com comprimento **diferente** e em pontos de tensão
**diferentes**. Subtração direta é impossível — não existe correspondência
ponto a ponto entre os dois vetores.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.plot(V10[:12],  Q10[:12],  'o-', ms=6, color='steelblue',
        label=f'Ciclo 10 ({len(V10)} pts)')
ax.plot(V100[:12], Q100[:12], 's--', ms=6, color='tomato',
        label=f'Ciclo 100 ({len(V100)} pts)')
ax.set_xlabel('Tensao (V)')
ax.set_ylabel('Capacidade (Ah)')
ax.set_title('Primeiros 12 pontos de cada ciclo\n(grades de tensao diferentes!)')
ax.legend()

ax = axes[1]
ax.plot(V10,  Q10,  color='steelblue', lw=1.5, alpha=0.85, label='Ciclo 10')
ax.plot(V100, Q100, color='tomato', lw=1.5, alpha=0.85, linestyle='--', label='Ciclo 100')
ax.set_xlabel('Tensao (V)')
ax.set_ylabel('Capacidade Q(V) (Ah)')
ax.set_title('Curvas brutas completas\n(impossivel subtrair diretamente)')
ax.legend()

plt.tight_layout()
plt.show()


## 3. A Solução: Spline + Grade Uniforme de 1.000 Pontos

O artigo resolve em duas etapas:

1. **Ajustar uma spline cúbica** que passa exatamente pelos pontos medidos
2. **Avaliar a spline** em 1.000 pontos uniformes entre 3,5 V e 2,0 V

Após isso, todos os ciclos têm exatamente 1.000 valores de Q nos
mesmos pontos de tensão — a subtração direta se torna válida.

> **Resolução:** (3,5 − 2,0) V ÷ 999 passos = **1,5 mV por ponto** —
> suficiente para capturar todos os detalhes das curvas.


In [ ]:
def padronizar_curva(V_bruto, Q_bruto, n_pontos=1000):
    """
    Padroniza uma curva de descarga para uma grade uniforme.
    1. Ordena os dados (spline precisa de x crescente)
    2. Remove pontos de tensao duplicados
    3. Ajusta spline cubica (s=0 = interpolacao exata)
    4. Avalia na grade uniforme de n_pontos
    """
    V_grid = np.linspace(3.5, 2.0, n_pontos)   # grade igual para TODOS

    V_ord = V_bruto[::-1].copy()   # inverte para crescente
    Q_ord = Q_bruto[::-1].copy()

    _, idx = np.unique(V_ord, return_index=True)   # remove duplicatas
    spline = UnivariateSpline(V_ord[idx], Q_ord[idx], s=0, ext='const')

    return V_grid, spline(V_grid)

V_grid, Q10_std  = padronizar_curva(V10,  Q10)
_,      Q100_std = padronizar_curva(V100, Q100)

print(f'Apos padronizacao:')
print(f'  Grade: {len(V_grid)} pontos de {V_grid[0]:.1f} V a {V_grid[-1]:.1f} V')
print(f'  Resolucao: {(V_grid[0]-V_grid[-1])/(len(V_grid)-1)*1000:.2f} mV por ponto')
print(f'  Ciclo  10: vetor de {len(Q10_std)} valores')
print(f'  Ciclo 100: vetor de {len(Q100_std)} valores')
print('Subtracao direta agora e valida!')


## 4. Resultado: Curvas na Mesma Grade e a Curva Diferencial

Com os dois ciclos padronizados, podemos:
- **Sobrepor** as curvas para comparação visual
- **Subtrair** para obter ΔQ(V) — a diferença ponto a ponto

O gráfico à direita mostra ΔQ₁₀₀₋₁₀(V): quanto a distribuição de capacidade
**mudou** em cada nível de tensão. Se a bateria estivesse saudável, essa
curva seria zero em toda a faixa.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ax = axes[0]
ax.plot(V_grid, Q10_std,  lw=2, color='steelblue', label='Ciclo 10')
ax.plot(V_grid, Q100_std, lw=2, color='tomato', linestyle='--', label='Ciclo 100')
ax.set_xlabel('Tensao (V)')
ax.set_ylabel('Capacidade Q(V) (Ah)')
ax.set_title('Curvas Q(V) padronizadas\n(mesma grade de 1.000 pontos)')
ax.legend()

ax = axes[1]
dQ = Q100_std - Q10_std
ax.plot(V_grid, dQ, lw=2, color='purple')
ax.axhline(0, color='black', lw=0.7, linestyle=':')
ax.fill_between(V_grid, dQ, 0, alpha=0.18, color='purple')
ax.set_xlabel('Tensao (V)')
ax.set_ylabel('DeltaQ_100-10(V) (Ah)')
ax.set_title(f'Curva diferencial Delta Q(V)\nVariancia = {np.var(dQ):.2e}  |  Minimo = {dQ.min():.4f} Ah')

plt.tight_layout()
plt.show()

print('Resumo:')
print('  Entrada:  curvas brutas com N pontos irregulares')
print('  Saida:    vetor Q(V) com 1.000 pontos na mesma grade de tensao')
print('  Permite:  subtracao direta — base para todas as features do artigo')


## 5. Resumo

| Etapa | O que faz | Por que |
|-------|-----------|---------|
| Spline | Ajusta curva suave aos pontos medidos | Permite avaliar em qualquer tensao |
| Grade uniforme | 1.000 tensoes fixas de 3,5 V a 2,0 V | Mesma grade para todos os ciclos |
| Q(V) ao inves de V(Q) | Tensao no eixo X, capacidade no Y | Faixa de tensao identica = grade compartilhada |
| DeltaQ(V) | Diferenca entre dois ciclos | Captura mudancas antes da queda de capacidade |

Proximo notebook: como usar Delta Q(V) para construir features que preveem a vida util.
